In [ ]:
import os
from pathlib import Path
# basically if you do Path("example/example2/example.txt") it turns smart and you can do stuff like directory.iterdir() which returns a container of everything in the directory
import numpy as np

import torch
import torch.nn as nn

import cv2
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from sklearn.model_selection import train_test_split

import torchvision.models.video as video_models
from torchvision.models.video import R2Plus1D_18_Weights

from collections import defaultdict

In [ ]:
datasetpath = "/kaggle/input/datasets/sarbagyashakya/basketball-51-dataset/Basketball_51 dataset"
frames = 16
samplesperclass = 9999 # set to 9999 or some big number for the full dataset

workers = 4
batchsize = 32

modelpath = "/kaggle/input/models/spicybigmac/basketballmodel/pytorch/default/1/testmodel.pt"
savedir = "/kaggle/working/"
os.makedirs(savedir, exist_ok=True)
bestpath = os.path.join(savedir, "bestmodel.pt")

epochs = 4
patience = 5

# determinism stuff
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True

In [ ]:
# we need to subtract the mean color and divide by the standard deviation of the Kinetics-400 dataset
# this makes it easier to train the model which is already used to these colors

traintrans = v2.Compose([
    v2.Resize([112,112], antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.43216, 0.394666, 0.37645], std=[0.22803, 0.22145, 0.216989])
])

testtrans = v2.Compose([
    v2.Resize([112,112], antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.43216, 0.394666, 0.37645], std=[0.22803, 0.22145, 0.216989])
])

class BasketballDataset(Dataset):
    def __init__(self, training, datadir, frames=16, samplesperclass=5):
        self.datadir = Path(datadir)
        self.goalframes = frames
        self.training = training
        
        # use sorted here to ensure determinism so the labels arent cooked if the drive is shuffled
        self.classes = sorted([d.name for d in self.datadir.iterdir() if d.is_dir() and not d.name[0] == "."])

        videopaths, labels = [], []

        for i,name in enumerate(self.classes):
            classdir = self.datadir / name

            # use sorted here to ensure determinism so i can debug a specific video index
            vids = sorted([f for f in classdir.iterdir() if f.is_file() and f.suffix.lower() == ".mp4"])

            for path in vids[:samplesperclass]:
                # this only takes the first samplesperclass videos from each class
                videopaths.append(path)
                labels.append(i)

        # self._cache = defaultdict() # cache for less cpu work
        
        # im js a 42-using sheep
        trainpaths, testpaths, trainlabels, testlabels = train_test_split(videopaths, labels, test_size=0.2, stratify=labels, random_state=42)

        if self.training:
            self.videopaths = trainpaths
            self.labels = trainlabels
        else:
            self.videopaths = testpaths
            self.labels = testlabels

        print(f"loaded {len(self.videopaths)} video paths for {'training' if training else 'testing'} :3")

    def __len__(self):
        # dataloader uses this function for like progress bars and stuff
        return len(self.videopaths)

    def __getitem__(self, idx):
        # dataloader always calls this function with idx as the only parameter
        path = self.videopaths[idx]
        label = self.labels[idx]

        frames = self._load_video(path)

        vidtensor = torch.from_numpy(frames)
        # the v2 transform function looks at the last 2 dimensions, so we need to make them height and width respectively
        vidtensor = vidtensor.permute(0,3,1,2)
        if self.training:
            vidtensor = traintrans(vidtensor)
        else:
            vidtensor = testtrans(vidtensor)
        vidtensor = vidtensor.permute(1,0,2,3)

        return vidtensor, label

    def _load_video(self, path):
        cap = cv2.VideoCapture(str(path))
        framecount = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if framecount < self.goalframes:
            # note: this will NEVER HAPPEN unless a video file gets corrupted
            idxs = np.pad(np.arange(framecount), (0, self.goalframes - framecount), mode='edge')
        else:
            idxs = np.linspace(0, framecount - 1, self.goalframes, dtype=int)

        idxs = set(idxs)
        frames = []
        frameidx = 0

        while len(frames) < self.goalframes:
            ret, frame = cap.read()

            if ret:
                if frameidx in idxs:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # convert color
                    frames.append(frame)
            else:
                # videos broken so extend the last working frame otherwise black screen
                while len(frames) < self.goalframes:
                    frames.append(frames[-1] if len(frames) > 0 else np.zeros((int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), 3), dtype=np.uint8))

            frameidx += 1

        cap.release()
        return np.stack(frames)

In [ ]:
traindata = BasketballDataset(True, datasetpath, frames, samplesperclass)
trainloader = DataLoader(traindata, batch_size=batchsize, shuffle=True, num_workers=workers, pin_memory=True, persistent_workers=True)

testdata = BasketballDataset(False, datasetpath, frames, samplesperclass)
testloader = DataLoader(testdata, batch_size=batchsize, shuffle=False, num_workers=workers, pin_memory=True, persistent_workers=True)
# no shuffle for test

classes = len(traindata.classes)

In [ ]:
# # loading the R(2+1)D-18 model
weights = video_models.R2Plus1D_18_Weights.KINETICS400_V1
model = video_models.r2plus1d_18(weights=weights)

infeatures = model.fc.in_features
model.fc = nn.Linear(infeatures, classes)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.device_count() > 1:
    print(f"using {torch.cuda.device_count()} GPUs bless kaggle :praying_hands:")
    model = nn.DataParallel(model)

model = model.to(device)

print(f"model loaded on {device}")

In [ ]:
idxtoclass = {i: v for i,v in enumerate(traindata.classes)}

criterion = nn.CrossEntropyLoss(reduction="none")
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0002, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.7, patience=0)

currbestacc = 0
stale = 0

In [ ]:
# training loop

# save the following epoch-wise data:
# training and testing loss and accuracy
# per class accuracy 
history = []

for epoch in range(1, epochs+1):
    print(f"\n\n\n\n\n==================== epoch {epoch}/{epochs} ====================")

    # training

    print("\n\n---------- TRAINING BEGINNING ----------")
    model.train()

    # trainloss separates the total loss by the true label
    trainloss, traincorrect, traintotal = defaultdict(float), defaultdict(float), defaultdict(float)
    totaltrainloss, totaltraincorrect, totaltrainsamples = 0,0,0

    j = 0
    for vids, labels in trainloader:
        vids, labels = vids.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(vids)
        loss = criterion(outputs, labels)

        loss.mean().backward()
        optimizer.step()

        # vids shape: [batch, rgb, frames, height, width]
        # labels shape: [batch]
        # outputs: [batch, classes]

        # tracking
        _, predicted = outputs.max(1)
        for i in range(labels.size(0)):
            label = labels[i].item() # .item() basically gets a normal python data type from a pytorch tensor
            pred = predicted[i].item()

            trainloss[label] += loss[i].item()
            traintotal[label] += 1
            traincorrect[label] += (label == pred)

            totaltraincorrect += (label == pred)

        totaltrainloss += loss.sum().item()
        totaltrainsamples += labels.size(0)

        j += 1
        if j%50 == 0 or j == len(trainloader):
            print(f"\nbatch {j}/{len(trainloader)} completed:")
            print("\nclass      loss      accuracy")
            for k in range(classes):
                acc = (traincorrect[k]/traintotal[k]) if traintotal[k] > 0 else 0
                avgloss = (trainloss[k]/traintotal[k]) if traintotal[k] > 0 else 0
                print(f"{idxtoclass[k]}:       {avgloss:.3f}     {acc:.1%}")

            print(f"\nrunning loss: {(totaltrainloss/totaltrainsamples):.3f}")
            print(f"\nrunning accuracy: {totaltraincorrect/totaltrainsamples:.1%}")


    # testing

    print("\n\n\n\n\n\n\n---------- TESTING BEGINNING ----------")
    model.eval()

    testloss, testcorrect, testtotal = defaultdict(float), defaultdict(float), defaultdict(float)
    totaltestloss, totaltestcorrect, totaltestsamples = 0, 0, 0

    with torch.no_grad():
        j = 0
        for vids, labels in testloader:
            vids, labels = vids.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            outputs = model(vids)
            loss = criterion(outputs, labels)

            # tracking
            _, predicted = outputs.max(1)
            for i in range(labels.size(0)):
                label = labels[i].item()
                pred = predicted[i].item()

                testloss[label] += loss[i].item()
                testtotal[label] += 1
                testcorrect[label] += (label == pred)

                totaltestcorrect += (label == pred)

            totaltestloss += loss.sum().item()
            totaltestsamples += labels.size(0)

            j += 1
            # if j%10 == 0 or j == 1 or j == len(testloader):
            if j == len(testloader):
                print(f"\nbatch {j}/{len(testloader)} completed:")
                print("\nclass      loss      accuracy")
                for k in range(classes):
                    acc = (testcorrect[k]/testtotal[k]) if testtotal[k] > 0 else 0
                    avgloss = (testloss[k]/testtotal[k]) if testtotal[k] > 0 else 0
                    print(f"{idxtoclass[k]}:       {avgloss:.3f}     {acc:.1%}")

                print(f"\nrunning test loss: {totaltestloss/totaltestsamples:.3f}")
                print(f"\nrunning test accuracy: {totaltestcorrect/totaltestsamples:.1%}")

    # learning rate scheduler
    oldlr = optimizer.param_groups[0]["lr"]
    scheduler.step(totaltestcorrect/totaltestsamples)
    if (optimizer.param_groups[0]["lr"] != oldlr):
        print(f"\nlearning rate reduced from {oldlr} to {optimizer.param_groups[0]["lr"]}")

    # add stuff to history
    history.append([totaltrainloss/totaltrainsamples, totaltraincorrect/totaltrainsamples, [trainloss[i]/traintotal[i] for i in range(classes)], [traincorrect[i]/traintotal[i] for i in range(classes)], totaltestloss/totaltestsamples, totaltestcorrect/totaltestsamples, [testloss[i]/testtotal[i] for i in range(classes)], [testcorrect[i]/testtotal[i] for i in range(classes)]])
    
    # model saving and checkpointing and accuracy stuff
    if totaltestcorrect/totaltestsamples > currbestacc:
        stale = 0
        currbestacc = totaltestcorrect/totaltestsamples
        torch.save(model.state_dict(), bestpath)
    else:
        stale += 1
        if stale >= patience:
            print(f"\n\nno improvement after {patience} epochs, stopping now.")
            break

        
# save history to json
import json

historypath = os.path.join(savedir, "history.json")
with open(historypath, "w") as f:
    json.dump(history, f)

In [ ]:
# visualizations

from matplotlib import pyplot as plt

# train and test loss over epochs

plt.plot([history[i][0] for i in range(len(history))], label="training", color="blue")
plt.plot([history[i][4] for i in range(len(history))], label="validation", color="red")

plt.xlabel("Epochs")
plt.ylabel("Average Loss")
plt.title("Average Loss per Epoch")

plt.legend()
plt.show()


# train and test accuracy over epochs

plt.plot([history[i][1] for i in range(len(history))], label="training", color="blue", ls="--")
plt.plot([history[i][5] for i in range(len(history))], label="validation", color="red", ls="--")

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy per Epoch")

plt.legend()
plt.show()


# loss of each class over epochs

colors = ["red","orange","yellow","green","blue","cyan","purple","black"]

# training
for j in range(classes):
    plt.plot([history[i][2][j] for i in range(len(history))], label=f"{traindata.classes[j]}", color=colors[j], ls="-")

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Average Loss per Class per Epoch (training)")

plt.legend()
plt.show()

# validation
for j in range(classes):
    plt.plot([history[i][6][j] for i in range(len(history))], label=f"{testdata.classes[j]}", color=colors[j], ls="--")

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Average Loss per Class per Epoch (validation)")

plt.legend()
plt.show()


# accuracy of each class over epochs

# training
for j in range(classes):
    plt.plot([history[i][3][j] for i in range(len(history))], label=f"{traindata.classes[j]}", color=colors[j])

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy per Class per Epoch (training)")

plt.legend()
plt.show()

# validation
for j in range(classes):
    plt.plot([history[i][7][j] for i in range(len(history))], label=f"{testdata.classes[j]}", color=colors[j])

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy per Class per Epoch (validation)")

plt.legend()
plt.show()